# Word embeddings: words as vectors

In Lecture 4 we used AI assistants built on **large language models (LLMs)**. A
foundational idea behind those models is the *word embedding*: every word is
represented as a vector of numbers, arranged so that words with similar meanings
end up close together in the vector space.

Here we explore a classic set of pre-trained embeddings called
[GloVe](https://nlp.stanford.edu/projects/glove/) (Global Vectors). Each word is
a point in a 50-dimensional space. We will see that simple **vector arithmetic**
on these points captures surprising amounts of meaning.

## Loading the vectors

Training embeddings needs a huge text corpus and a lot of compute, so we use
vectors that were trained once and shared. To keep the download small, we ship a
subset of the 20,000 most common words as `glove_subset.npz` alongside this
notebook.

In [ ]:
import numpy as np

data = np.load("glove_subset.npz")
vocab = list(data["vocab"])              # list of 20,000 words
# Stored as float16 to keep the file small; use float32 for the maths.
vectors = data["vectors"].astype(np.float32)   # shape (20000, 50)

word_to_index = {word: i for i, word in enumerate(vocab)}

print("number of words:", len(vocab))
print("vector shape:", vectors.shape)

Each word maps to one 50-dimensional vector. Let's look at the vector for `rome`:

In [ ]:
vectors[word_to_index["rome"]]

In [ ]:
vectors[word_to_index["rome"]].shape

## Finding similar words

A common way to measure how similar two vectors are is the **cosine similarity**:
the cosine of the angle between them. It is 1.0 for identical directions and near
0 for unrelated ones. We normalize every vector to unit length and then a
similarity is just a dot product.

The function below returns the words whose vectors are most similar to a query
vector.

In [ ]:
# Pre-normalize all vectors to unit length so a dot product gives cosine similarity.
norms = np.linalg.norm(vectors, axis=1, keepdims=True)
unit_vectors = vectors / norms


def most_similar(query_vector, topn=5, exclude=()):
    """Return the `topn` (word, similarity) pairs closest to `query_vector`."""
    q = query_vector / np.linalg.norm(query_vector)
    similarities = unit_vectors @ q
    ranking = np.argsort(-similarities)
    results = []
    for i in ranking:
        word = vocab[i]
        if word in exclude:
            continue
        results.append((word, float(similarities[i])))
        if len(results) >= topn:
            break
    return results


most_similar(vectors[word_to_index["rome"]])

The nearest neighbours of `rome` are other Italian and European cities. Try `dog`:

In [ ]:
most_similar(vectors[word_to_index["dog"]])

## Analogies through vector arithmetic

The famous result from word embeddings is that relationships between words show up
as *directions* in the vector space. The direction from `man` to `king` is roughly
the same as the direction from `woman` to `queen`. So we can ask

> king − man + woman = ?

and expect to land near `queen`.

In [ ]:
def analogy(a, b, c):
    """Solve 'a is to b as c is to ?'  ->  vector(b) - vector(a) + vector(c)."""
    vec = (vectors[word_to_index[b]]
           - vectors[word_to_index[a]]
           + vectors[word_to_index[c]])
    return most_similar(vec, topn=3, exclude={a, b, c})


# man is to king as woman is to ...?
analogy("man", "king", "woman")

In [ ]:
# italy is to rome as china is to ...?
analogy("italy", "rome", "china")

In [ ]:
# germany is to berlin as italy is to ...?
analogy("germany", "berlin", "italy")

## Why this matters

These embeddings were trained with a simple objective: predict which words appear
near each other in text. Nothing told the model that Rome is the capital of Italy
or that kings are male. That structure *emerged* from statistics of language.

Modern LLMs (the transformers behind the assistants from Lecture 4) start from the
same idea but go much further: instead of one fixed vector per word, they compute
vectors that depend on the surrounding context, stacked through many layers. The
intuition to carry forward is the same one you just saw here — **meaning can be
represented as geometry in a high-dimensional space.**